In [ ]:
import os                                     
import tensorflow as tf
from tensorflow.keras import layers, models          # type: ignore 
from tensorflow.keras.layers import Dropout          # type: ignore 
from tensorflow.keras.preprocessing.image import ImageDataGenerator          # type: ignore 

# For Model 3 onwards 
from tensorflow.keras.callbacks import EarlyStopping                         # type: ignore 

# For Model 4
from tensorflow.keras.callbacks import ReduceLROnPlateau                     # type: ignore
from tensorflow.keras.layers import BatchNormalization                       # type: ignore
from tensorflow.keras.callbacks import ModelCheckpoint                       # type: ignore

# SECTION 1 — TRAIN DATA AUGMENTATION : “image transformation machine” ----->  image processing pipeline ban rahi hai.

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.1,
    horizontal_flip=True,
    width_shift_range=0.2,
    height_shift_range=0.2,

    # Model 4 onwards
    brightness_range=[0.8,1.2],
    shear_range=0.1,
    fill_mode='nearest'
    )

val_datagen = ImageDataGenerator(rescale=1./255)

# SECTION 3: TRAIN GENERATOR
train_generator = train_datagen.flow_from_directory(
    'C:/Users/sanch/Documents/Git_hub/Streamlit-KrishiNetra/02_dataset/01_train',
    target_size=(180,180),
    batch_size=12,
    class_mode='categorical',
    shuffle=True
)

# SECTION 4: VALIDATION GENERATOR
val_generator = val_datagen.flow_from_directory(
    'C:/Users/sanch/Documents/Git_hub/Streamlit-KrishiNetra/02_dataset/02_validate',
    target_size=(180,180),
    batch_size=12,
    class_mode='categorical',
    shuffle=False
)


# 3. Build CNN Model 4 :

model = models.Sequential()

#-----------------------------------------------------------------------------------------------------------------------#

# C1: Convolution
model.add(layers.Conv2D(filters=32, kernel_size=(3,3), activation='relu', padding='same', input_shape=(180,180,3)))  #padding='same' ----> preserve edge information

model.add(BatchNormalization())

# C2: Convolution
model.add(layers.Conv2D(filters=32, kernel_size=(3,3), activation='relu', padding='same'))  #padding='same' ----> preserve edge information

model.add(BatchNormalization())

# P1: Max Pooling
model.add(layers.MaxPooling2D(pool_size=(2,2)))


#-----------------------------------------------------------------------------------------------------------------------#

# C3: Convolution
model.add(layers.Conv2D(filters=64, kernel_size=(3,3), activation='relu', padding='same'))  #padding='same' ----> preserve edge information

model.add(BatchNormalization())

# C4: Convolution
model.add(layers.Conv2D(filters=64, kernel_size=(3,3), activation='relu', padding='same'))

model.add(BatchNormalization())

# P2: Max Pooling
model.add(layers.MaxPooling2D(pool_size=(2,2)))

#-----------------------------------------------------------------------------------------------------------------------#

# C5: Convolution
model.add(layers.Conv2D(filters= 128, kernel_size=(3,3), activation='relu', padding='same'))

model.add(BatchNormalization())

# C6: Convolution
model.add(layers.Conv2D(filters=128, kernel_size=(3,3), activation='relu', padding='same'))

model.add(BatchNormalization())

# P3: Max Pooling
model.add(layers.MaxPooling2D(pool_size=(2,2)))

#-----------------------------------------------------------------------------------------------------------------------#

# Global Feature Aggregation
model.add(layers.GlobalAveragePooling2D())    # GlobalAveragePooling ----->reduce memorization

#-----------------------------------------------------------------------------------------------------------------------#

# Fully Connected Layers
model.add(layers.Dense(128, activation='relu'))
model.add(Dropout(0.3))                            # stronger dropout---------->better generalization
model.add(layers.Dense(64, activation='relu'))
model.add(Dropout(0.3))

Found 18085 images belonging to 19 classes.
Found 4515 images belonging to 19 classes.


c:\Users\sanch\Documents\Git_hub\Streamlit-KrishiNetra\krishi_env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [3]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 180, 180, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 180, 180, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 180, 180, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 180, 180, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 90, 90, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 90, 90, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 90, 90, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 90, 90, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 90, 90, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 45, 45, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 45, 45, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 45, 45, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 45, 45, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 45, 45, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 22, 22, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 313,568 (1.20 MB)

 Trainable params: 312,672 (1.19 MB)

 Non-trainable params: 896 (3.50 KB)

In [ ]:
# 4. Compile Model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])



early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True)

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    verbose=1
)

checkpoint = ModelCheckpoint(
    '04_models/SK_plant_model_v4_pro.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

# 5. TRAINING

print("\n Training started ........")
history = model.fit(
    train_generator,
    validation_data = val_generator,
    epochs=20,
    callbacks=[early_stop, lr_scheduler, checkpoint]
)
